In [2]:
import pandas as pd

df = pd.read_excel("screening_results_scopus.xlsx", engine="openpyxl")

# Step 1: keep only excluded papers
excluded = df[df["decision"] == "exclude"].copy()

# Step 2: keep only those with valid (non-empty) reasons
def has_valid_reason(x):
    if pd.isna(x):
        return False
    x = str(x).strip().lower()
    return x != "" and "unspecified" not in x and "error" not in x

excluded_valid = excluded[excluded["matched_exclusion_criteria"].apply(has_valid_reason)]

print("Excluded papers (total):", len(excluded))
print("Excluded with clear reasons:", len(excluded_valid))
print("Excluded without clear reasons:", len(excluded) - len(excluded_valid))

Excluded papers (total): 139
Excluded with clear reasons: 138
Excluded without clear reasons: 1


In [3]:
# Get papers WITHOUT clear reasons
excluded_no_reason = excluded[~excluded["matched_exclusion_criteria"].apply(has_valid_reason)]

print("\n--- Papers with NO clear exclusion reason ---\n")

for i, row in excluded_no_reason.iterrows():
    print(f"Title: {row['title']}")
    print(f"Abstract: {row['abstract']}")
    print("-" * 80)


--- Papers with NO clear exclusion reason ---

Title: Robotic tactile perception and understanding: A sparse coding method
Abstract: This book introduces the challenges of robotic tactile perception and task understanding, and describes an advanced approach based on machine learning and sparse coding techniques. Further, a set of structured sparse coding models is developed to address the issues of dynamic tactile sensing. The book then proves that the proposed framework is effective in solving the problems of multi-finger tactile object recognition, multi-label tactile adjective recognition and multi-category material analysis, which are all challenging practical problems in the fields of robotics and automation. The proposed sparse coding model can be used to tackle the challenging visual-tactile fusion recognition problem, and the book develops a series of efficient optimization algorithms to implement the model. It is suitable as a reference book for graduate students with a basic

In [4]:
from collections import Counter

all_reasons = []

for r in excluded_valid["matched_exclusion_criteria"]:
    parts = [x.strip() for x in str(r).split(";") if x.strip()]
    all_reasons.extend(parts)

counts = Counter(all_reasons)

print("\nReason counts:")
for k, v in counts.items():
    print(k, ":", v)

print("\nSum of reason counts:", sum(counts.values()))


Reason counts:
the abstract does not clearly involve both visual and tactile/haptic information : 40
the task is not related to perception or recognition : 37
the main focus is grasping, grasp planning, manipulation, robot control, trajectory planning, or pose estimation : 28
the paper is a review, survey, tutorial, editorial, or non-original study : 30
abstract does not clearly involve both visual and tactile/haptic information : 9
the paper is mainly about teleoperation, VR user study, haptic rendering, or human perception without machine perception : 31
task not related to perception or recognition : 1
the paper is mainly about human perception without machine perception : 3
the main focus is not on perception-level tasks related to visuo-haptic/visuo-tactile perception : 1
focus on immersive experience modeling for education rather than perception tasks : 1
not primarily about perception or recognition tasks : 1
the paper is mainly about remote sensing tasks : 1
the paper is mainl

In [6]:
import pandas as pd

df = pd.read_excel("screening_results_scopus.xlsx", engine="openpyxl")

# keep only excluded
excluded = df[df["decision"] == "exclude"].copy()

# count number of reasons per paper
def count_reasons(x):
    if pd.isna(x):
        return 0
    return len([r for r in str(x).split(";") if r.strip()])

excluded["num_reasons"] = excluded["matched_exclusion_criteria"].apply(count_reasons)

# papers with multiple reasons
multi_reason = excluded[excluded["num_reasons"] > 1]

print("Number of papers with multiple exclusion reasons:", len(multi_reason))

# show them
for i, row in multi_reason.iterrows():
    print("\n---")
    print("Title:", row["title"])
    print("Reasons:", row["matched_exclusion_criteria"])

Number of papers with multiple exclusion reasons: 48

---
Title: Creating a framework for holistic assessment of aesthetics: A response to nilsson and axelsson (2015) on attributes of aesthetic quality of textile quality
Reasons: the abstract does not clearly involve both visual and tactile/haptic information; the task is not related to perception or recognition

---
Title: Creating Tactile Interaction Surfaces for the Origo Steering Wheel Concept using CWI and EHWs
Reasons: the paper is a review, survey, tutorial, editorial, or non-original study; the abstract does not clearly involve both visual and tactile/haptic information

---
Title: Musical-Space Synaesthesia: Visualisation of Musical Texture
Reasons: the paper is mainly about human perception without machine perception; the abstract does not clearly involve both visual and tactile/haptic information; the task is not related to perception or recognition

---
Title: Collection of Design Directions for the Realization of a Visual 